# SoundStream training on Kaggle

**Before running:**
1. Add dataset [LibriSpeech](https://www.kaggle.com/datasets/a24998667/librispeech) (or update `LIBRI_INPUT` below).
2. For a private repo add Kaggle secret `GITHUB_TOKEN`. Public repo clones without a token.
3. Enable GPU.

Trains 100 epochs with `baseline_kaggle`. Checkpoint: `saved/v4-kaggle-100ep/checkpoint-epoch100.pth`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import shutil
import time
import torch

REPO_DIR = "soundstream_hw"
REPO_URL = "https://github.com/ndrew1337/soundstream_hw.git"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = f"https://{token}@github.com/ndrew1337/soundstream_hw.git"
except Exception:
    pass

subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
if str(Path(REPO_DIR).resolve()) not in sys.path:
    sys.path.insert(0, str(Path(REPO_DIR).resolve()))

In [ ]:
os.environ["PIP_NO_WARN_CONFLICTS"] = "1"
!pip install -q torchmetrics pystoi "numba>=0.59" librosa soundfile requests tqdm wget matplotlib pandas wandb hydra-core omegaconf

In [ ]:
LIBRI_INPUT = "/kaggle/input/datasets/a24998667/librispeech"

os.makedirs("/kaggle/working/lsdata", exist_ok=True)
for part in ["train-clean-100", "test-clean"]:
    src = f"{LIBRI_INPUT}/{part}"
    dst = f"/kaggle/working/lsdata/{part}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
!ls /kaggle/working/lsdata/

## Train

In [ ]:
t0 = time.time()
!WANDB_MODE=disabled HYDRA_FULL_ERROR=1 python train.py -cn baseline_kaggle \
    datasets.train.data_dir=/kaggle/working/lsdata \
    writer.run_name=v4-kaggle-100ep \
    writer.mode=disabled \
    trainer.override=True
print(f"Elapsed: {(time.time() - t0) / 3600:.2f} h")

## Inference 

In [ ]:
CKPT = "saved/v4-kaggle-100ep/checkpoint-epoch100.pth"
if not Path(CKPT).is_file():
    raise FileNotFoundError(f"Train first or set CKPT. Missing: {CKPT}")

!WANDB_MODE=disabled HYDRA_FULL_ERROR=1 python inference.py \
    datasets.test.data_dir=/kaggle/working/lsdata \
    inferencer.from_pretrained={CKPT} \
    inferencer.save_path=test-clean-final

In [ ]:
psrc = Path("saved/v4-kaggle-100ep/checkpoint-epoch100.pth")
out_dir = Path("/kaggle/working/output")
out_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(src, out_dir / "checkpoint-epoch100.pth")
print("Saved to", out_dir / "checkpoint-epoch100.pth")